In [4]:
# Ring 1: Single tool, single turn.
# Source for <CodeSource> in build-a-tool-using-agent.mdx.

import json
import anthropic
from rich.pretty import pprint

# Create a client. It reads ANTHROPIC_API_KEY from the environment.
client = anthropic.Anthropic()

# Define one tool. The input_schema is a JSON Schema object describing
# the arguments Claude should pass when it calls this tool. This schema
# includes nested objects (recurrence), arrays (attendees), and optional
# fields, which is closer to real-world tools than a flat string argument.
tools = [
    {
        "name": "create_calendar_event",
        "description": "Create a calendar event with attendees and optional recurrence.",
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "start": {"type": "string", "format": "date-time"},
                "end": {"type": "string", "format": "date-time"},
                "attendees": {
                    "type": "array",
                    "items": {"type": "string", "format": "email"},
                },
                "recurrence": {
                    "type": "object",
                    "properties": {
                        "frequency": {"enum": ["daily", "weekly", "monthly"]},
                        "count": {"type": "integer", "minimum": 1},
                    },
                },
            },
            "required": ["title", "start", "end"],
        },
    }
]

# Send the user's request along with the tool definition. Claude decides
# whether to call the tool based on the request and the tool description.
response = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1024,
    tools=tools,
    tool_choice={"type": "auto", "disable_parallel_tool_use": True},
    messages=[
        {
            "role": "user",
            "content": "Schedule a 30-minute sync with alice@example.com and bob@example.com next Monday at 10am.",
        }
    ],
)

# When Claude calls a tool, the response has stop_reason "tool_use"
# and the content array contains a tool_use block alongside any text.
pprint(response)

# Find the tool_use block. A response may contain text blocks before the
# tool_use block, so scan the content array rather than assuming position.
tool_use = next(block for block in response.content if block.type == "tool_use")
pprint(tool_use)

# Execute the tool. In a real system this would call your calendar API.
# Here the result is hardcoded to keep the example self-contained.
result = {"event_id": "evt_123", "status": "created"}

# Send the result back. The tool_result block goes in a user message and
# its tool_use_id must match the id from the tool_use block above. The
# assistant's previous response is included so Claude has the full history.
followup = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1024,
    tools=tools,
    tool_choice={"type": "auto", "disable_parallel_tool_use": True},
    messages=[
        {
            "role": "user",
            "content": "Schedule a 30-minute sync with alice@example.com and bob@example.com next Monday at 10am.",
        },
        {"role": "assistant", "content": response.content},
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use.id,
                    "content": json.dumps(result),
                }
            ],
        },
    ],
)

# With the tool result in hand, Claude produces a final natural-language
# answer and stop_reason becomes "end_turn".
pprint(followup)

Message(
│   id='msg_01J6bZxejyd9PRCqvUhC551R',
│   container=None,
│   content=[
│   │   TextBlock(
│   │   │   citations=None,
│   │   │   text="\n\nI'll schedule that 30-minute sync for next Monday at 10am. Let me create the event now.",
│   │   │   type='text'
│   │   ),
│   │   ToolUseBlock(
│   │   │   id='toolu_01DmJ4Lm6D9fsfHNEKvz5dna',
│   │   │   caller=DirectCaller(type='direct'),
│   │   │   input={
│   │   │   │   'title': 'Sync',
│   │   │   │   'start': '2025-07-14T10:00:00',
│   │   │   │   'end': '2025-07-14T10:30:00',
│   │   │   │   'attendees': ['alice@example.com', 'bob@example.com']
│   │   │   },
│   │   │   name='create_calendar_event',
│   │   │   type='tool_use'
│   │   )
│   ],
│   model='claude-opus-4-6',
│   role='assistant',
│   stop_details=None,
│   stop_reason='tool_use',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=499,
│   │   output_tokens=160,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)

ToolUseBlock(
│   id='toolu_01DmJ4Lm6D9fsfHNEKvz5dna',
│   caller=DirectCaller(type='direct'),
│   input={
│   │   'title': 'Sync',
│   │   'start': '2025-07-14T10:00:00',
│   │   'end': '2025-07-14T10:30:00',
│   │   'attendees': ['alice@example.com', 'bob@example.com']
│   },
│   name='create_calendar_event',
│   type='tool_use'
)

Message(
│   id='msg_01VJZKyhWR1GKmzYbXHMceWw',
│   container=None,
│   content=[
│   │   TextBlock(
│   │   │   citations=None,
│   │   │   text="Your meeting has been scheduled! Here's a summary:\n\n- **Title:** Sync\n- **Date:** Monday, July 14, 2025\n- **Time:** 10:00 AM – 10:30 AM\n- **Attendees:** alice@example.com, bob@example.com\n\nBoth attendees should receive an invitation. Let me know if you'd like to make any changes!",
│   │   │   type='text'
│   │   )
│   ],
│   model='claude-opus-4-6',
│   role='assistant',
│   stop_details=None,
│   stop_reason='end_turn',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=697,
│   │   output_tokens=92,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)

In [6]:
# Ring 2: The agentic loop.
# Source for <CodeSource> in build-a-tool-using-agent.mdx.

import json

import anthropic

client = anthropic.Anthropic()

tools = [
    {
        "name": "create_calendar_event",
        "description": "Create a calendar event with attendees and optional recurrence.",
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "start": {"type": "string", "format": "date-time"},
                "end": {"type": "string", "format": "date-time"},
                "attendees": {
                    "type": "array",
                    "items": {"type": "string", "format": "email"},
                },
                "recurrence": {
                    "type": "object",
                    "properties": {
                        "frequency": {"enum": ["daily", "weekly", "monthly"]},
                        "count": {"type": "integer", "minimum": 1},
                    },
                },
            },
            "required": ["title", "start", "end"],
        },
    }
]


def run_tool(name, tool_input):
    if name == "create_calendar_event":
        return {"event_id": "evt_123", "status": "created", "title": tool_input["title"]}
    return {"error": f"Unknown tool: {name}"}


# Keep the full conversation history in a list so each turn sees prior context.
messages = [
    {
        "role": "user",
        "content": "Schedule a weekly team standup every Monday at 9am for the next 4 weeks. Invite the whole team: alice@example.com, bob@example.com, carol@example.com.",
    }
]

response = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1024,
    tools=tools,
    tool_choice={"type": "auto", "disable_parallel_tool_use": True},
    messages=messages,
)
pprint(response)

# Loop until Claude stops asking for tools. Each iteration runs the requested
# tool, appends the result to history, and asks Claude to continue.
while response.stop_reason == "tool_use":
    tool_use = next(block for block in response.content if block.type == "tool_use")
    result = run_tool(tool_use.name, tool_use.input)

    messages.append({"role": "assistant", "content": response.content})
    messages.append(
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use.id,
                    "content": json.dumps(result),
                }
            ],
        }
    )

    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=1024,
        tools=tools,
        tool_choice={"type": "auto", "disable_parallel_tool_use": True},
        messages=messages,
    )

final_text = next(block for block in response.content if block.type == "text")
pprint(response)

Message(
│   id='msg_01M3ZXSLqd3V6yzQ31W1Diz2',
│   container=None,
│   content=[
│   │   TextBlock(
│   │   │   citations=None,
│   │   │   text="\n\nI'll schedule the weekly team standup for you right away!",
│   │   │   type='text'
│   │   ),
│   │   ToolUseBlock(
│   │   │   id='toolu_01RBj8MX2kQ1ofocUdp3Pvhu',
│   │   │   caller=DirectCaller(type='direct'),
│   │   │   input={
│   │   │   │   'title': 'Team Standup',
│   │   │   │   'start': '2025-07-07T09:00:00',
│   │   │   │   'end': '2025-07-07T09:30:00',
│   │   │   │   'attendees': ['alice@example.com', 'bob@example.com', 'carol@example.com'],
│   │   │   │   'recurrence': {'frequency': 'weekly', 'count': 4}
│   │   │   },
│   │   │   name='create_calendar_event',
│   │   │   type='tool_use'
│   │   )
│   ],
│   model='claude-opus-4-6',
│   role='assistant',
│   stop_details=None,
│   stop_reason='tool_use',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=517,
│   │   output_tokens=186,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)

Message(
│   id='msg_01AbNx2Ma6Kvc48KNBDaoPxu',
│   container=None,
│   content=[
│   │   TextBlock(
│   │   │   citations=None,
│   │   │   text="Your **Team Standup** has been scheduled! Here's a summary:\n\n- **📅 When:** Every Monday at 9:00 AM (starting July 7, 2025)\n- **🔁 Recurrence:** Weekly for 4 weeks (July 7, 14, 21, 28)\n- **⏱️ Duration:** 30 minutes\n- **👥 Attendees:**\n  - alice@example.com\n  - bob@example.com\n  - carol@example.com\n\nAll three team members will receive an invite. Let me know if you'd like to adjust anything, such as the duration or add more attendees!",
│   │   │   type='text'
│   │   )
│   ],
│   model='claude-opus-4-6',
│   role='assistant',
│   stop_details=None,
│   stop_reason='end_turn',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=750,
│   │   output_tokens=156,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)

In [8]:
# Ring 3: Multiple tools, parallel calls.
# Source for <CodeSource> in build-a-tool-using-agent.mdx.

import json

import anthropic

client = anthropic.Anthropic()

tools = [
    {
        "name": "create_calendar_event",
        "description": "Create a calendar event with attendees and optional recurrence.",
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "start": {"type": "string", "format": "date-time"},
                "end": {"type": "string", "format": "date-time"},
                "attendees": {
                    "type": "array",
                    "items": {"type": "string", "format": "email"},
                },
                "recurrence": {
                    "type": "object",
                    "properties": {
                        "frequency": {"enum": ["daily", "weekly", "monthly"]},
                        "count": {"type": "integer", "minimum": 1},
                    },
                },
            },
            "required": ["title", "start", "end"],
        },
    },
    {
        "name": "list_calendar_events",
        "description": "List all calendar events on a given date.",
        "input_schema": {
            "type": "object",
            "properties": {
                "date": {"type": "string", "format": "date"},
            },
            "required": ["date"],
        },
    },
]


def run_tool(name, tool_input):
    if name == "create_calendar_event":
        return {"event_id": "evt_123", "status": "created", "title": tool_input["title"]}
    if name == "list_calendar_events":
        return {"events": [{"title": "Existing meeting", "start": "14:00", "end": "15:00"}]}
    return {"error": f"Unknown tool: {name}"}


messages = [
    {
        "role": "user",
        "content": "Check what I have next Monday, then schedule a planning session that avoids any conflicts.",
    }
]

response = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1024,
    tools=tools,
    messages=messages,
)
pprint(response)

while response.stop_reason == "tool_use":
    # A single response can contain multiple tool_use blocks. Process all of
    # them and return all results together in one user message.
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            result = run_tool(block.name, block.input)
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                }
            )

    messages.append({"role": "assistant", "content": response.content})
    messages.append({"role": "user", "content": tool_results})

    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=1024,
        tools=tools,
        messages=messages,
    )

final_text = next(block for block in response.content if block.type == "text")
pprint(messages)

Message(
│   id='msg_01FCsQa4PoTKJ7nrhvHRmtN6',
│   container=None,
│   content=[
│   │   TextBlock(
│   │   │   citations=None,
│   │   │   text="\n\nI'll start by checking your calendar for next Monday to see what's already scheduled.",
│   │   │   type='text'
│   │   ),
│   │   ToolUseBlock(
│   │   │   id='toolu_01KXqWJnWDkbiJpG8FZTh3Z8',
│   │   │   caller=DirectCaller(type='direct'),
│   │   │   input={'date': '2025-07-14'},
│   │   │   name='list_calendar_events',
│   │   │   type='tool_use'
│   │   )
│   ],
│   model='claude-opus-4-6',
│   role='assistant',
│   stop_details=None,
│   stop_reason='tool_use',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=763,
│   │   output_tokens=79,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)

[
│   {
│   │   'role': 'user',
│   │   'content': 'Check what I have next Monday, then schedule a planning session that avoids any conflicts.'
│   },
│   {
│   │   'role': 'assistant',
│   │   'content': [
│   │   │   TextBlock(
│   │   │   │   citations=None,
│   │   │   │   text="\n\nI'll start by checking your calendar for next Monday to see what's already scheduled.",
│   │   │   │   type='text'
│   │   │   ),
│   │   │   ToolUseBlock(
│   │   │   │   id='toolu_01KXqWJnWDkbiJpG8FZTh3Z8',
│   │   │   │   caller=DirectCaller(type='direct'),
│   │   │   │   input={'date': '2025-07-14'},
│   │   │   │   name='list_calendar_events',
│   │   │   │   type='tool_use'
│   │   │   )
│   │   ]
│   },
│   {
│   │   'role': 'user',
│   │   'content': [
│   │   │   {
│   │   │   │   'type': 'tool_result',
│   │   │   │   'tool_use_id': 'toolu_01KXqWJnWDkbiJpG8FZTh3Z8',
│   │   │   │   'content': '{"events": [{"title": "Existing meeting", "start": "14:00", "end": "15:00"}]}'
│   │   │   }
│   │   ]
│   },
│   {
│   │   'role': 'assistant',
│   │   'content': [
│   │   │   TextBlock(
│   │   │   │   citations=None,
│   │   │   │   text="Here's what you have on **Monday, July 14th**:\n\n- **Existing meeting** — 2:00 PM – 3:00 PM\n\nTo avoid that conflict, I'll schedule your **Planning Session** in the morning. How about **10:00 AM – 11:00 AM**? Let me go ahead and create that:",
│   │   │   │   type='text'
│   │   │   ),
│   │   │   ToolUseBlock(
│   │   │   │   id='toolu_01CPypviAVeKuapo3XQbnnj8',
│   │   │   │   caller=DirectCaller(type='direct'),
│   │   │   │   input={'title': 'Planning Session', 'start': '2025-07-14T10:00:00', 'end': '2025-07-14T11:00:00'},
│   │   │   │   name='create_calendar_event',
│   │   │   │   type='tool_use'
│   │   │   )
│   │   ]
│   },
│   {
│   │   'role': 'user',
│   │   'content': [
│   │   │   {
│   │   │   │   'type': 'tool_result',
│   │   │   │   'tool_use_id': 'toolu_01CPypviAVeKuapo3XQbnnj8',
│   │   │   │   'content': '{"event_id": "evt_123", "status": "created", "title": "Planning Session"}'
│   │   │   }
│   │   ]
│   }
]

In [10]:
# Ring 4: Error handling.
# Source for <CodeSource> in build-a-tool-using-agent.mdx.

import json
import anthropic

client = anthropic.Anthropic()

tools = [
    {
        "name": "create_calendar_event",
        "description": "Create a calendar event with attendees and optional recurrence.",
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "start": {"type": "string", "format": "date-time"},
                "end": {"type": "string", "format": "date-time"},
                "attendees": {
                    "type": "array",
                    "items": {"type": "string", "format": "email"},
                },
                "recurrence": {
                    "type": "object",
                    "properties": {
                        "frequency": {"enum": ["daily", "weekly", "monthly"]},
                        "count": {"type": "integer", "minimum": 1},
                    },
                },
            },
            "required": ["title", "start", "end"],
        },
    },
    {
        "name": "list_calendar_events",
        "description": "List all calendar events on a given date.",
        "input_schema": {
            "type": "object",
            "properties": {
                "date": {"type": "string", "format": "date"},
            },
            "required": ["date"],
        },
    },
]


def run_tool(name, tool_input):
    if name == "create_calendar_event":
        if "attendees" in tool_input and len(tool_input["attendees"]) > 10:
            raise ValueError("Too many attendees (max 10)")
        return {"event_id": "evt_123", "status": "created", "title": tool_input["title"]}
    if name == "list_calendar_events":
        return {"events": [{"title": "Existing meeting", "start": "14:00", "end": "15:00"}]}
    raise ValueError(f"Unknown tool: {name}")


messages = [
    {
        "role": "user",
        "content": "Schedule next Monday at 10am an all-hands with everyone: " + ", ".join(f"user{i}@example.com" for i in range(15)),
    }
]

response = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1024,
    tools=tools,
    messages=messages,
)
pprint(response)

while response.stop_reason == "tool_use":
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            try:
                result = run_tool(block.name, block.input)
                tool_results.append(
                    {"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(result)}
                )
            except Exception as exc:
                # Signal failure so Claude can retry or ask for clarification.
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(exc),
                        "is_error": True,
                    }
                )

    messages.append({"role": "assistant", "content": response.content})
    messages.append({"role": "user", "content": tool_results})

    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=1024,
        tools=tools,
        messages=messages,
    )

final_text = next(block for block in response.content if block.type == "text")
pprint(messages)

Message(
│   id='msg_014qw8ogdV8po1moZjuxwLiD',
│   container=None,
│   content=[
│   │   TextBlock(
│   │   │   citations=None,
│   │   │   text="\n\nI'll schedule the all-hands meeting for next Monday at 10am. Let me create that event with all 15 attendees.",
│   │   │   type='text'
│   │   ),
│   │   ToolUseBlock(
│   │   │   id='toolu_01L2aqoFkwQeW6ADY9FGXoZr',
│   │   │   caller=DirectCaller(type='direct'),
│   │   │   input={
│   │   │   │   'title': 'All-Hands Meeting',
│   │   │   │   'start': '2025-07-28T10:00:00',
│   │   │   │   'end': '2025-07-28T11:00:00',
│   │   │   │   'attendees': [
│   │   │   │   │   'user0@example.com',
│   │   │   │   │   'user1@example.com',
│   │   │   │   │   'user2@example.com',
│   │   │   │   │   'user3@example.com',
│   │   │   │   │   'user4@example.com',
│   │   │   │   │   'user5@example.com',
│   │   │   │   │   'user6@example.com',
│   │   │   │   │   'user7@example.com',
│   │   │   │   │   'user8@example.com',
│   │   │   │   │   'user9@example.com',
│   │   │   │   │   'user10@example.com',
│   │   │   │   │   'user11@example.com',
│   │   │   │   │   'user12@example.com',
│   │   │   │   │   'user13@example.com',
│   │   │   │   │   'user14@example.com'
│   │   │   │   ]
│   │   │   },
│   │   │   name='create_calendar_event',
│   │   │   type='tool_use'
│   │   )
│   ],
│   model='claude-opus-4-6',
│   role='assistant',
│   stop_details=None,
│   stop_reason='tool_use',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=863,
│   │   output_tokens=283,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)

[
│   {
│   │   'role': 'user',
│   │   'content': 'Schedule next Monday at 10am an all-hands with everyone: user0@example.com, user1@example.com, user2@example.com, user3@example.com, user4@example.com, user5@example.com, user6@example.com, user7@example.com, user8@example.com, user9@example.com, user10@example.com, user11@example.com, user12@example.com, user13@example.com, user14@example.com'
│   },
│   {
│   │   'role': 'assistant',
│   │   'content': [
│   │   │   TextBlock(
│   │   │   │   citations=None,
│   │   │   │   text="\n\nI'll schedule the all-hands meeting for next Monday at 10am. Let me create that event with all 15 attendees.",
│   │   │   │   type='text'
│   │   │   ),
│   │   │   ToolUseBlock(
│   │   │   │   id='toolu_01L2aqoFkwQeW6ADY9FGXoZr',
│   │   │   │   caller=DirectCaller(type='direct'),
│   │   │   │   input={
│   │   │   │   │   'title': 'All-Hands Meeting',
│   │   │   │   │   'start': '2025-07-28T10:00:00',
│   │   │   │   │   'end': '2025-07-28T11:00:00',
│   │   │   │   │   'attendees': [
│   │   │   │   │   │   'user0@example.com',
│   │   │   │   │   │   'user1@example.com',
│   │   │   │   │   │   'user2@example.com',
│   │   │   │   │   │   'user3@example.com',
│   │   │   │   │   │   'user4@example.com',
│   │   │   │   │   │   'user5@example.com',
│   │   │   │   │   │   'user6@example.com',
│   │   │   │   │   │   'user7@example.com',
│   │   │   │   │   │   'user8@example.com',
│   │   │   │   │   │   'user9@example.com',
│   │   │   │   │   │   'user10@example.com',
│   │   │   │   │   │   'user11@example.com',
│   │   │   │   │   │   'user12@example.com',
│   │   │   │   │   │   'user13@example.com',
│   │   │   │   │   │   'user14@example.com'
│   │   │   │   │   ]
│   │   │   │   },
│   │   │   │   name='create_calendar_event',
│   │   │   │   type='tool_use'
│   │   │   )
│   │   ]
│   },
│   {
│   │   'role': 'user',
│   │   'content': [
│   │   │   {
│   │   │   │   'type': 'tool_result',
│   │   │   │   'tool_use_id': 'toolu_01L2aqoFkwQeW6ADY9FGXoZr',
│   │   │   │   'content': 'Too many attendees (max 10)',
│   │   │   │   'is_error': True
│   │   │   }
│   │   ]
│   },
│   {
│   │   'role': 'assistant',
│   │   'content': [
│   │   │   TextBlock(
│   │   │   │   citations=None,
│   │   │   │   text="It looks like there's a limit of 10 attendees per event. To work around this, I'll split the group across two linked events:",
│   │   │   │   type='text'
│   │   │   ),
│   │   │   ToolUseBlock(
│   │   │   │   id='toolu_01NKgowEYQpCyRhk8hR4ycUW',
│   │   │   │   caller=DirectCaller(type='direct'),
│   │   │   │   input={
│   │   │   │   │   'title': 'All-Hands Meeting (Group 1)',
│   │   │   │   │   'start': '2025-07-28T10:00:00',
│   │   │   │   │   'end': '2025-07-28T11:00:00',
│   │   │   │   │   'attendees': [
│   │   │   │   │   │   'user0@example.com',
│   │   │   │   │   │   'user1@example.com',
│   │   │   │   │   │   'user2@example.com',
│   │   │   │   │   │   'user3@example.com',
│   │   │   │   │   │   'user4@example.com',
│   │   │   │   │   │   'user5@example.com',
│   │   │   │   │   │   'user6@example.com',
│   │   │   │   │   │   'user7@example.com',
│   │   │   │   │   │   'user8@example.com',
│   │   │   │   │   │   'user9@example.com'
│   │   │   │   │   ]
│   │   │   │   },
│   │   │   │   name='create_calendar_event',
│   │   │   │   type='tool_use'
│   │   │   ),
│   │   │   ToolUseBlock(
│   │   │   │   id='toolu_013Pg4hKzqfCN5RyGSjNVKZR',
│   │   │   │   caller=DirectCaller(type='direct'),
│   │   │   │   input={
│   │   │   │   │   'title': 'All-Hands Meeting (Group 2)',
│   │   │   │   │   'start': '2025-07-28T10:00:00',
│   │   │   │   │   'end': '2025-07-28T11:00:00',
│   │   │   │   │   'attendees': [
│   │   │   │   │   │   'user10@example.com',
│   │   │   │   │   │   'user11@example.com',
│   │   │   │   │   │   'user12@example.com',
│   │   │   │   │   │   'user13@example.com',
│   │   │   │   │   │   'user14@example.com'
│   │   │   │   │   ]


In [11]:
# Ring 5: The Tool Runner SDK abstraction.
# Source for <CodeSource> in build-a-tool-using-agent.mdx.

import json
import anthropic
from anthropic import beta_tool

client = anthropic.Anthropic()


@beta_tool
def create_calendar_event(
    title: str,
    start: str,
    end: str,
    attendees: list[str] | None = None,
    recurrence: dict | None = None,
) -> str:
    """Create a calendar event with attendees and optional recurrence.

    Args:
        title: Event title.
        start: Start time in ISO 8601 format.
        end: End time in ISO 8601 format.
        attendees: Email addresses to invite.
        recurrence: Dict with 'frequency' (daily, weekly, monthly) and 'count'.
    """
    if attendees and len(attendees) > 10:
        raise ValueError("Too many attendees (max 10)")
    return json.dumps({"event_id": "evt_123", "status": "created", "title": title})


@beta_tool
def list_calendar_events(date: str) -> str:
    """List all calendar events on a given date.

    Args:
        date: Date in YYYY-MM-DD format.
    """
    return json.dumps({"events": [{"title": "Existing meeting", "start": "14:00", "end": "15:00"}]})


final_message = client.beta.messages.tool_runner(
    model="claude-opus-4-6",
    max_tokens=1024,
    tools=[create_calendar_event, list_calendar_events],
    messages=[
        {
            "role": "user",
            "content": "Check what I have next Monday, then schedule a planning session that avoids any conflicts.",
        }
    ],
).until_done()

for block in final_message.content:
    if block.type == "text":
        print(block.text)

All set! Here's your updated Monday schedule:

| Time | Event |
|---|---|
| **10:00 AM – 11:00 AM** | ✅ Planning Session *(just added)* |
| **2:00 PM – 3:00 PM** | Existing meeting |

No conflicts! If you'd like to adjust the time, add attendees, or make any other changes, just let me know.
